In [1]:
from mlflow.tracking import MlflowClient

MLFLOW_TRACKING_URI = "sqlite:///mlflow.db"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [2]:
# client.create_experiment(name="my-cool-experiment")

In [3]:
from mlflow.entities import ViewType

runs = client.search_runs(
    experiment_ids='1',
    filter_string='metrics.rmse < 5.81',
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=10,
    order_by=['metrics.rmse ASC']
)

In [4]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: 8580c0ae9f9343aeaeec1da87fa3c359, rmse: 5.7940
run id: 1ca0ed933cb34a63b6f12958b9c47a5f, rmse: 5.7940
run id: 33b6b0c5431149328a2d08dec0fec403, rmse: 5.8070
run id: 9bf2259365ed4f6eb3f1a1309ddda5c5, rmse: 5.8074
run id: d3157b3d3148426592b37ac24311885d, rmse: 5.8088
run id: f4087a6d42cb4f61937cf34e4f3c3f2c, rmse: 5.8089
run id: a557e0064d7641c0ab2b64b867181331, rmse: 5.8092
run id: a4eda0255998453ca23011bf90cd3f3d, rmse: 5.8097


In [5]:
import mlflow

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [6]:
run_id = "9bf2259365ed4f6eb3f1a1309ddda5c5" # third on list
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name="nyc-taxi-regressor-programatically")

Registered model 'nyc-taxi-regressor-programatically' already exists. Creating a new version of this model...
Created version '2' of model 'nyc-taxi-regressor-programatically'.


<ModelVersion: aliases=[], creation_timestamp=1748217289888, current_stage='None', description=None, last_updated_timestamp=1748217289888, name='nyc-taxi-regressor-programatically', run_id='9bf2259365ed4f6eb3f1a1309ddda5c5', run_link=None, source='/workspaces/mlops-zoomcamp/02-experiment_tracking/mlruns/1/9bf2259365ed4f6eb3f1a1309ddda5c5/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=2>

In [7]:
client.list_artifacts("cc94b933bf604d228324d181e179aca6") # registered model

[<FileInfo: file_size=6034, is_dir=False, path='feature_importance_weight.json'>,
 <FileInfo: file_size=269894, is_dir=False, path='feature_importance_weight.png'>,
 <FileInfo: file_size=None, is_dir=True, path='model'>]

In [8]:
model_name = "nyc-taxi-regressor"
client.get_latest_versions(name=model_name)

/tmp/ipykernel_7520/1481491681.py:2: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.get_latest_versions(name=model_name)


[<ModelVersion: aliases=[], creation_timestamp=1748205624313, current_stage='None', description='', last_updated_timestamp=1748205624313, name='nyc-taxi-regressor', run_id='8580c0ae9f9343aeaeec1da87fa3c359', run_link='', source='/workspaces/mlops-zoomcamp/02-experiment_tracking/mlruns/1/8580c0ae9f9343aeaeec1da87fa3c359/artifacts/models_mlflow', status='READY', status_message=None, tags={}, user_id=None, version=3>,
 <ModelVersion: aliases=[], creation_timestamp=1748202227256, current_stage='Staging', description=('This model version was transitioned to Staging even though MLFlow has '
  'deprecated that functionality'), last_updated_timestamp=1748203640390, name='nyc-taxi-regressor', run_id='cc94b933bf604d228324d181e179aca6', run_link='', source='/workspaces/mlops-zoomcamp/02-experiment_tracking/mlruns/1/cc94b933bf604d228324d181e179aca6/artifacts/model', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=2>]

In [9]:
client.transition_model_version_stage(
    name=model_name,
    version=2,
    stage="Staging",
    archive_existing_versions=False
)

/tmp/ipykernel_7520/1872662155.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1748202227256, current_stage='Staging', description=('This model version was transitioned to Staging even though MLFlow has '
 'deprecated that functionality'), last_updated_timestamp=1748217289950, name='nyc-taxi-regressor', run_id='cc94b933bf604d228324d181e179aca6', run_link='', source='/workspaces/mlops-zoomcamp/02-experiment_tracking/mlruns/1/cc94b933bf604d228324d181e179aca6/artifacts/model', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=2>

In [10]:
client.update_model_version(
    name=model_name,
    version=2,
    description="This model version was transitioned to Staging even though MLFlow has deprecated that functionality"
)

<ModelVersion: aliases=[], creation_timestamp=1748202227256, current_stage='Staging', description=('This model version was transitioned to Staging even though MLFlow has '
 'deprecated that functionality'), last_updated_timestamp=1748217289971, name='nyc-taxi-regressor', run_id='cc94b933bf604d228324d181e179aca6', run_link='', source='/workspaces/mlops-zoomcamp/02-experiment_tracking/mlruns/1/cc94b933bf604d228324d181e179aca6/artifacts/model', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=2>

In [11]:
from sklearn.metrics import mean_squared_error
import pandas as pd

def read_clean_df(filename):
    df = pd.read_parquet(filename)
    
    df['lpep_dropoff_datetime'] = pd.to_datetime(df['lpep_dropoff_datetime'])
    df['lpep_pickup_datetime'] = pd.to_datetime(df['lpep_pickup_datetime'])
    
    df['duration'] = df['lpep_dropoff_datetime'] - df['lpep_pickup_datetime']
    df['duration'] = df['duration'].apply(lambda x: x.total_seconds() / 60)
    
    df = df.loc[((df.duration >= 1) & (df.duration <= 60))]
    
    cat_feats = ['PULocationID', 'DOLocationID']
    df[cat_feats] = df[cat_feats].astype(str)
    
    return df

def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    cat_feats = ['PU_DO']
    train_dicts = df[cat_feats].to_dict(orient='records')
    return dv.transform(train_dicts)

def test_model(stage, X_test, y_test, name='nyc-taxi-regressor'):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(X_test)
    return {"rmse": mean_squared_error(y_test, y_pred)**(0.5)}

In [12]:
df = read_clean_df('../data/green_tripdata_2023-03.parquet')

In [13]:
run_id="8580c0ae9f9343aeaeec1da87fa3c359"
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/workspaces/mlops-zoomcamp/02-experiment_tracking/preprocessor'

In [14]:
import pickle

with open("preprocessor/preprocessor.b", 'rb') as f_in:
    dv= pickle.load(f_in)

In [15]:
X_test = preprocess(df, dv)

In [16]:
target = "duration"
y_test = df[target].values

In [17]:
test_model(name=model_name, stage='3', X_test=X_test, y_test=y_test)

{'rmse': 10.555933455781684}